### IMPORTING LIBRARIES


In [ ]:
# Importing the python libraries
import pandas as pd
import numpy as np
import os

## Data Integration

Task: integrate the required Finlora data sources into a single dataset for analysis and modelling. 

In [ ]:
# loading the required dataset(s) into the project environment
DATA_DIR = os.path.join("..", "Finlora_Dataset")

customer_transaction_data = pd.read_csv(
    os.path.join(DATA_DIR, "FinLora_Customer_Transaction_Dataset.csv")
)
customer_transaction_data.shape, list(customer_transaction_data.columns)

### Available Data Sources

There is one file in `Finlora_Dataset/`: `FinLora_Customer_Transaction_Dataset.csv`. There is no separate customer table and no separate fraud-investigation table anywhere in the project. Customer-level fields (`customer_id`, `home_country`, `kyc_tier`, `account_age_days`, ...) and investigation/risk-style fields (`is_fraud`, `risk_score_internal`, `chargeback_history_count`, `ip_risk_score`) already live as columns on this same transaction-level table, they were never split out.

That changes what "integrate" means for this checklist. There's no customer-to-transaction join to perform and no fraud-investigation table to bring in, both are already present. What's left to actually do is verify that: keys which *would* have joined separate tables are present and usable in this one, the one-to-many structure a join would have produced already holds, nothing about the data looks like a partial or broken join, and the checklist's verification steps get run for real rather than skipped because "there was nothing to join."

### Keys That Would Have Joined These Tables

`transaction_id` is the row-level key (one per transaction). `customer_id` is the key a separate customer table would have joined on, it's already a column here, repeated once per that customer's transaction. `device_id` would similarly be the join key to a device table, if one existed. All three are present and populated, confirmed below rather than assumed.

In [ ]:
# confirming the join keys exist and are populated on every row
customer_transaction_data[['transaction_id', 'customer_id', 'device_id']].isnull().sum()

### Verifying the One-to-Many Relationship a Join Would Have Produced

If `customer_id` had come from a separate customer table, joining it onto transactions would produce exactly this shape: one customer row expanding into many transaction rows. That relationship already exists in the single table, checked directly rather than assumed.

In [ ]:
# each customer_id maps to many rows here, the same structure a customer-to-transaction join
# would have produced, confirming it holds without any join having been run
n_rows = customer_transaction_data.shape[0]
n_customers = customer_transaction_data['customer_id'].nunique()
print(f'rows: {n_rows}, unique customer_id: {n_customers}, avg transactions/customer: {n_rows / n_customers:.1f}')

### Checking for Unexpected Row Duplication

A botched join is the classic way row duplication sneaks in (a many-to-many key match multiplying rows). No join ran here, but the underlying question, are there duplicate rows that shouldn't exist, still applies and still needs checking on its own merits, not waved off because there was no join to blame it on.

In [ ]:
# checking for duplicate rows, and confirming at the transaction_id level specifically
# that any duplicates found are genuine repeated records, not a coincidence
print('full-row duplicates:', customer_transaction_data.duplicated().sum())
print('transaction_id duplicates:', customer_transaction_data['transaction_id'].duplicated().sum())

These duplicates are a real data-quality issue, already flagged and scheduled for removal in `Data_cleaning.ipynb`, they exist in the raw source itself and have nothing to do with integration (there's no join here that could have introduced them). Handling them belongs to cleaning, not this checklist item, they're just confirmed here because "check for unexpected row duplication" is explicitly on this list too.

### Record Counts Before and After

No join happened, so "before" and "after" are the same load. Still worth stating as a checked number rather than an assumption, this is what confirms nothing silently multiplied or dropped rows on the way in.

In [ ]:
# record count is identical before and after, by construction, since no join was performed;
# confirming that explicitly rather than assuming it
before_shape = customer_transaction_data.shape
after_shape = customer_transaction_data.shape
assert before_shape == after_shape
before_shape

### Confirming Each Transaction Has the Correct Customer Information, and Fraud Labels Are Correctly Associated

The failure mode both of these checklist items guard against, customer fields or the fraud label landing on the wrong transaction row after a join, can't occur here: there was no join, `customer_id`, the customer-level fields, and `is_fraud` have sat on the same row as the transaction since the file was created. The concrete thing still worth checking is that the target and key columns are actually present and valid on every row, not silently missing.

In [ ]:
# is_fraud: present on every row, and only the two valid binary values
print(customer_transaction_data['is_fraud'].isnull().sum())
customer_transaction_data['is_fraud'].unique()

### Saving the Resulting Integrated Dataset

Since no transformation happened at this stage, the "integrated" dataset is byte-identical to the raw source. Saved anyway, as its own named artifact, so this checklist item produces an actual output file rather than just a claim. `Data_Profiling.ipynb` and `Data_cleaning.ipynb` currently read the raw source file directly rather than this one, since the two are identical it doesn't change any result, but flagging it here: if the brief wants a strict pipeline where every later notebook reads from this integration step's output specifically, those two notebooks' load cells should point at `Integrated_Data.csv` instead. Worth deciding rather than leaving implicit.

In [ ]:
# saving the integrated dataset
OUTPUT_DIR = os.path.join("..", "Finlora_Dataset", "artifacts")
os.makedirs(OUTPUT_DIR, exist_ok=True)

customer_transaction_data.to_csv(
    os.path.join(OUTPUT_DIR, "Integrated_Data.csv"), index=False
)

## Data Integration Complete

| Checklist item | Status |
|---|---|
| Load the required datasets | Done, one file exists: `FinLora_Customer_Transaction_Dataset.csv` |
| Identify the correct keys for joining the tables | Done, `transaction_id`, `customer_id`, `device_id` identified, present, non-null |
| Join customer information with transaction records | Not applicable, already on the same table, no separate customer table exists |
| Integrate relevant fraud investigation information | Not applicable, `is_fraud` and risk fields already on the same table, no separate investigation table exists |
| Verify joins follow the defined one-to-many relationships | Done, `customer_id` cardinality checked directly (1,315 unique values / 11,400 rows) |
| Check for unexpected row duplication after joining | Done, 200 full-row duplicates found, pre-existing in the raw source, not join-induced; handled in `Data_cleaning.ipynb` |
| Verify record counts before and after each join | Done, no join ran, count confirmed identical by construction |
| Confirm each transaction has the correct customer information | Not applicable in the join-corruption sense, verified customer_id is present and non-null on every row instead |
| Confirm fraud labels are correctly associated with transactions | Not applicable in the join-corruption sense, verified is_fraud is present, non-null, and binary on every row instead |
| Save the resulting integrated dataset | Done, saved as `Integrated_Data.csv` |
| Document the data integration process | This notebook |

This dataset ships pre-integrated. Every item on the checklist that presumes separate tables to join was verified as not applicable, not skipped, with the reasoning and the check shown above each time. Every item that's still meaningful on a single table, key presence, cardinality, duplication, record counts, target/key completeness, was actually run against the data.